In [1]:
import pandas as pd
import numpy as np

# ==========================================
# 1. LOAD REFERENCE TABLES
# ==========================================
df_ilce_ref = pd.read_excel("data/ilce_koordinat.xlsx")
df_trafik   = pd.read_excel("data/trafik.xlsx")

# ==========================================
# 2. DISTRICT WASTE DATA (DYNAMIC)
# ==========================================
df_ilceler_raw = pd.read_excel("data/ilceler.xlsx")
df_ilceler_raw.columns = df_ilceler_raw.columns.astype(str).str.strip()
available_years = [y for y in ['2021','2022','2023','2024','2025']
                 if y in df_ilceler_raw.columns]

df_ilceler_dynamic = pd.DataFrame({
    'Ilce_Adi':      df_ilceler_raw['İlçe'].astype(str).str.strip().str.title(),
    'Yillik_Tonaj':  df_ilceler_raw[available_years].mean(axis=1)
})
df_ilceler = pd.merge(df_ilce_ref, df_ilceler_dynamic, on='Ilce_Adi', how='left')
df_ilceler['Yillik_Tonaj'] = df_ilceler['Yillik_Tonaj'].fillna(
    df_ilceler['Yillik_Tonaj'].mean())

# ==========================================
# 3. STATION DATA AND CAPACITY CALCULATION
# ==========================================
df_istasyonlar_raw = pd.read_excel("data/istasyonlar.xlsx")
df_istasyonlar = pd.DataFrame({
    'Istasyon_Adi': df_istasyonlar_raw['AKTARMA İSTASYONLARI'].astype(str).str.strip(),
    'Enlem':        df_istasyonlar_raw['LATITUDE'],
    'Boylam':       df_istasyonlar_raw['LONGITUDE'],
    'Alan_m2':      df_istasyonlar_raw['YÜZÖLÇÜMÜ (m2)']
})

Q_TOTAL = 12097.8
df_istasyonlar['Gunluk_Kapasite'] = (
    df_istasyonlar['Alan_m2'] / df_istasyonlar['Alan_m2'].sum()) * Q_TOTAL

asian_station_names = [
    "Küçük Bakkalköy Katı Atık Aktarma İstasyonu",
    "Hekimbaşı Katı Atık Aktama İstasyonu",
    "Aydınlı Katı Atık Aktarma İstasyonu",
    "Şile Katı Atık Aktarma İstasyonu"
]
df_istasyonlar['Yaka'] = np.where(
    df_istasyonlar['Istasyon_Adi'].isin(asian_station_names), 'Asya', 'Avrupa')

# ==========================================
# 4. d_i NORMALIZATION
# Raw IBB data: sum(d_i) = 17,269 ton/day
# System capacity (Q_total) = 12,097.8 ton/day
# Normalize to satisfy constraint that all waste must reach stations.
# ==========================================
d_i_raw = (df_ilceler['Yillik_Tonaj'] / 365.0).to_numpy()
d_i_normalized = d_i_raw * (Q_TOTAL / d_i_raw.sum())
df_ilceler['Gunluk_Tonaj'] = d_i_normalized

print(f"✅ Data loaded successfully!")
print(f"   Number of districts   : {len(df_ilceler)}")
print(f"   Number of stations    : {len(df_istasyonlar)}")
print(f"   Raw daily waste       : {d_i_raw.sum():.1f} ton/day")
print(f"   Normalized daily waste: {d_i_normalized.sum():.1f} ton/day")
print(f"   Total capacity        : {Q_TOTAL:.1f} ton/day")
print(f"\nStation Capacities:")
for _, row in df_istasyonlar.iterrows():
    short_name = row['Istasyon_Adi'].split()[0]
    print(f"  {short_name:<15} {row['Gunluk_Kapasite']:>7.1f} ton/day  [{row['Yaka']}]")

✅ Data loaded successfully!
   Number of districts   : 39
   Number of stations    : 9
   Raw daily waste       : 17269.1 ton/day
   Normalized daily waste: 12097.8 ton/day
   Total capacity        : 12097.8 ton/day

Station Capacities:
  Baruthane         340.4 ton/day  [Avrupa]
  Yenibosna        1872.2 ton/day  [Avrupa]
  Başakşehir       1491.4 ton/day  [Avrupa]
  Silivri           678.0 ton/day  [Avrupa]
  Hasdal           1531.8 ton/day  [Avrupa]
  Küçük            1134.7 ton/day  [Asya]
  Hekimbaşı        1702.0 ton/day  [Asya]
  Aydınlı          2723.2 ton/day  [Asya]
  Şile              624.1 ton/day  [Asya]


In [ ]:
import numpy as np
from scipy.optimize import minimize
from scipy.interpolate import CubicSpline
import folium
import ipywidgets as widgets
from IPython.display import display
import time

# ==========================================
# 1. FETCH DATA FROM CELL 1
# ==========================================
N_DISTRICTS  = len(df_ilceler)
N_STATIONS   = len(df_istasyonlar)
N_ROUTES     = N_DISTRICTS * N_STATIONS   # = 39 × 9 = 351

districts_list   = df_ilceler['Ilce_Adi'].tolist()
d_i              = df_ilceler['Gunluk_Tonaj'].to_numpy()   # normalized
asian_districts  = set(df_ilceler[df_ilceler['Yaka'] == 'Asya']['Ilce_Adi'])
districts_coords = list(zip(df_ilceler['Enlem'], df_ilceler['Boylam']))

station_names  = df_istasyonlar['Istasyon_Adi'].tolist()
Q_j            = df_istasyonlar['Gunluk_Kapasite'].to_numpy()
asian_stat_set = set(df_istasyonlar[df_istasyonlar['Yaka'] == 'Asya']['Istasyon_Adi'])
station_coords = list(zip(df_istasyonlar['Enlem'], df_istasyonlar['Boylam']))

D = np.load("data/distance_matrix.npy")
smooth_traffic_curve = CubicSpline(df_trafik["Saat"], df_trafik["Hiz_kmh"])


# ==========================================
# 2. SIDE CHECK (Bosphorus Crossing Matrix)
# ==========================================
def build_crossing_mask():
    """True = wrong side (penalize), False = same side (allow)"""
    mask = np.zeros((N_DISTRICTS, N_STATIONS), dtype=bool)
    for i, d_name in enumerate(districts_list):
        for j, s_name in enumerate(station_names):
            if (d_name in asian_districts) != (s_name in asian_stat_set):
                mask[i, j] = True
    return mask

CROSSING_MASK = build_crossing_mask()


# ==========================================
# 3. PROPOSAL OBJECTIVE FUNCTION
# Proposal Equation (2):
# min Σ_ij D_ij(1 + α·τ(t_ij)²)·x_ij  +  μ·Σ_j[max(0, Σ_i x_ij - Q_j)]²
# ==========================================
def build_penalized_D(CROSSING_PENALTY):
    D_pen = D.copy()
    D_pen[CROSSING_MASK] *= CROSSING_PENALTY
    return D_pen

def objective_function(z, D_penalized, ALPHA, MU):
    x   = z[:N_ROUTES].reshape(N_DISTRICTS, N_STATIONS)
    t   = z[N_ROUTES:].reshape(N_DISTRICTS, N_STATIONS)
    # Traffic congestion index τ(t)
    v_t = smooth_traffic_curve(t)
    tau = np.clip(1.0 - (v_t - 27.0) / 9.0, 0.0, 1.0)
    # Transport cost
    transport_cost = np.sum(D_penalized * (1.0 + ALPHA * tau**2) * x)
    # Capacity overflow penalty (Proposal Equation 2)
    excess         = np.maximum(0.0, np.sum(x, axis=0) - Q_j)
    capacity_pen   = MU * np.sum(excess**2)
    return (transport_cost + capacity_pen) / 10000.0

def equality_constraint(z):
    """Σ_j x_ij = d_i  (all waste must be transported daily)"""
    x = z[:N_ROUTES].reshape(N_DISTRICTS, N_STATIONS)
    return np.sum(x, axis=1) - d_i


# ==========================================
# 4. CROSSING-AWARE PRUNING (Post-Processing)
# Proposal Section F + G pruning logic:
# For each district, select largest x_ij, zero out smaller routes.
# Additional rule: selected station must be on same side as district.
# ==========================================
def crossing_aware_pruning(x_opt_raw, t_opt_raw, d_i):
    """
    Assign each district to station with highest allocation (argmax).
    If wrong side selected → switch to best same-side alternative.
    """
    N_D, N_S = x_opt_raw.shape
    x_clean  = np.zeros_like(x_opt_raw)
    t_clean  = np.zeros_like(t_opt_raw)

    for i, d_name in enumerate(districts_list):
        scores    = x_opt_raw[i].copy()
        is_d_asia = d_name in asian_districts

        # Zero out wrong-side stations
        for j, s_name in enumerate(station_names):
            if (s_name in asian_stat_set) != is_d_asia:
                scores[j] = -np.inf

        # Edge case: no same-side station available
        if np.all(np.isinf(scores)):
            scores = x_opt_raw[i].copy()   # allow crossing if necessary

        best_j = int(np.argmax(scores))
        x_clean[i, best_j] = d_i[i]
        t_clean[i, best_j] = t_opt_raw[i, best_j]

    return x_clean, t_clean


# ==========================================
# 5. MAIN OPTIMIZATION FUNCTION
# ==========================================
def run_optimization(ALPHA, MU, CROSSING_PENALTY):
    global x_opt, t_opt
    D_penalized = build_penalized_D(CROSSING_PENALTY)

    # Initial point: each district → nearest same-side station
    x0_matrix = np.zeros((N_DISTRICTS, N_STATIONS))
    for i, d_name in enumerate(districts_list):
        # Block wrong-side stations initially
        costs = D_penalized[i].copy()
        x0_matrix[i, int(np.argmin(costs))] = d_i[i]

    z0 = np.concatenate([x0_matrix.flatten(), np.full(N_ROUTES, 14.0)])

    print(f"⚙️  Running SLSQP [α={ALPHA}, μ={MU:.0f}, crossing_penalty={CROSSING_PENALTY:.0f}x]...")
    t_start = time.time()

    result = minimize(
        objective_function, z0,
        args=(D_penalized, ALPHA, MU),
        method='SLSQP',
        bounds=[(0, None)] * N_ROUTES + [(6.0, 22.0)] * N_ROUTES,
        constraints={'type': 'eq', 'fun': equality_constraint},
        options={'maxiter': 1000, 'ftol': 1e-5, 'eps': 5e-4, 'disp': False}
    )

    elapsed = time.time() - t_start
    print(f"   Duration: {elapsed:.1f} sec  |  Status: {'✅ Success' if result.success else '⚠️ ' + result.message}")

    x_opt_raw = result.x[:N_ROUTES].reshape(N_DISTRICTS, N_STATIONS)
    t_opt_raw = result.x[N_ROUTES:].reshape(N_DISTRICTS, N_STATIONS)

    # ---- Post-Processing: Crossing-Aware Pruning ----
    x_opt, t_opt = crossing_aware_pruning(x_opt_raw, t_opt_raw, d_i)
    # ======================================================
    # POST-PROCESSING 1: Capacity-aware rebalancing
    # Moves districts FROM overloaded stations to under-loaded ones.
    # ======================================================
    from src.solvers import capacity_aware_rebalancing
    _dy = df_ilceler['Yaka'].to_numpy()
    _sy = df_istasyonlar['Yaka'].to_numpy()
    x_opt = capacity_aware_rebalancing(
        x_opt, Q_j, D_penalized, _dy, _sy,
        N_DISTRICTS, N_STATIONS, capacity_tolerance=1.05
    )

    # ======================================================
    # POST-PROCESSING 2: Fill empty stations (inline)
    # ======================================================
    _x = x_opt.copy()
    for _j in range(N_STATIONS):
        _loads = np.sum(_x, axis=0)
        if _loads[_j] > 0.1:
            continue
        _jside = _sy[_j]
        _cands = []
        for _i in range(N_DISTRICTS):
            if _dy[_i] != _jside:
                continue
            _cj = int(np.argmax(_x[_i, :]))
            if _x[_i, _cj] < 0.1:
                continue
            if int(np.sum(_x[:, _cj] > 0.1)) < 2:
                continue
            _cands.append((D_penalized[_i, _j], _i, _cj))
        if not _cands:
            continue
        _cands.sort()
        _, _im, _jd = _cands[0]
        _ton = _x[_im, _jd]
        _x[_im, _jd] = 0.0
        _x[_im, _j]  = _ton
    x_opt = _x
    _empty = int(np.sum(np.sum(x_opt, axis=0) < 1.0))

    # ======================================================
    # POST-PROCESSING 3: Fleet allocation
    # 82 trucks total (Asia: 43, Europe: 39).
    # Allocate proportionally by route tonnage (largest-remainder method).
    # ======================================================
    K_ASIA, K_EU = 43, 39
    _station_loads = np.sum(x_opt, axis=0)
    fleet_alloc   = np.zeros(N_DISTRICTS, dtype=int)
    for _side, _K in [('Asya', K_ASIA), ('Avrupa', K_EU)]:
        _idx = [_i for _i in range(N_DISTRICTS) if df_ilceler['Yaka'].iloc[_i] == _side]
        if not _idx:
            continue
        _tons = np.array([d_i[_i] for _i in _idx])
        _quota = _K * _tons / _tons.sum()
        _base  = np.floor(_quota).astype(int)
        _rem   = _quota - _base
        _short = _K - _base.sum()
        _top   = np.argsort(_rem)[::-1][:_short]
        _base[_top] += 1
        for _k, _ii in enumerate(_idx):
            fleet_alloc[_ii] = _base[_k]
    print(f'\n  Fleet: Asia={K_ASIA} trucks, Europe={K_EU} trucks')
    print(f'  Truck range per route: {fleet_alloc.min()} - {fleet_alloc.max()}')
    print(f'  Total trucks: {fleet_alloc.sum()}\n')

    # ---- Results Summary ----
    station_loads = np.sum(x_opt, axis=0)
    overload      = float(np.sum(np.maximum(0.0, station_loads - Q_j)))
    n_cross       = sum(
        1 for i, d_name in enumerate(districts_list)
        for j in range(N_STATIONS)
        if x_opt[i, j] > 0.1 and CROSSING_MASK[i, j]
    )

    print(f"\n📊 RESULTS")
    print(f"   Active Routes     : {int(np.sum(x_opt > 0.1))} (1 route per district)")
    print(f"   Capacity Overflow : {overload:.1f} tons  (ideal: 0)")
    print(f"   Bosphorus Crossings: {n_cross} routes  (ideal: 0)")
    print(f"\n{'Station':<18} {'Capacity':>9} {'Load':>9} {'Utilization':>9} {'Districts':>6}")
    print("-" * 60)
    for j in range(N_STATIONS):
        pct    = station_loads[j] / Q_j[j] * 100
        status = '⚠️' if pct > 110 else '✅'
        short_name = station_names[j].split()[0]
        n_districts = int(np.sum(x_opt[:, j] > 0.1))
        print(f"  {status} {short_name:<15} {Q_j[j]:>8.0f}  {station_loads[j]:>8.0f}  {pct:>7.1f}%  {n_districts:>5}")

    # ---- Map ----
    m = folium.Map(location=[41.0082, 28.9784], zoom_start=10, tiles="CartoDB dark_matter")

    for idx, coord in enumerate(station_coords):
        pct   = station_loads[idx] / Q_j[idx] * 100
        color = 'red' if pct > 110 else ('orange' if pct > 90 else 'blue')
        short_name = station_names[idx].split()[0]
        folium.Marker(
            location=coord,
            popup=folium.Popup(
                f"<b>{short_name}</b> [{df_istasyonlar.iloc[idx]['Yaka']}]<br>"
                f"Capacity: {Q_j[idx]:.0f} ton/day<br>"
                f"Load: {station_loads[idx]:.0f} tons ({pct:.1f}%)",
                max_width=220
            ),
            icon=folium.Icon(color=color, icon='trash', prefix='fa')
        ).add_to(m)

    for i, d_name in enumerate(districts_list):
        for j in range(N_STATIONS):
            tonnage = x_opt[i, j]
            if tonnage < 0.5:
                continue
            t_val  = t_opt[i, j]
            hours  = int(t_val)
            mins   = int((t_val - hours) * 60)
            is_cross = CROSSING_MASK[i, j]
            color  = '#ff4444' if is_cross else '#00ff88'  # red=wrong side!

            # District point
            folium.CircleMarker(
                location=districts_coords[i], radius=4,
                color='#ffffff', fill=True,
                fill_color='#aaffcc', fill_opacity=0.9,
                popup=folium.Popup(f"<b>{d_name}</b><br>"
                    f"Waste: {tonnage:.1f} ton/day<br>"
                    f"Departure: {hours:02d}:{mins:02d}", max_width=200)
            ).add_to(m)

            folium.PolyLine(
                locations=[districts_coords[i], station_coords[j]],
                color=color,
                weight=max(1.5, tonnage / 35.0),
                opacity=0.75,
                tooltip=(
                    f"{'⚠️ BOSPHORUS CROSSING! ' if is_cross else ''}"
                    f"<b>{d_name}</b> → {station_names[j].split()[0]}<br>"
                    f"Distance: {D[i,j]:.1f} km | Tonnage: {tonnage:.1f} tons<br>"
                    f"Fleet Allocated: {fleet_alloc[i]} trucks<br>"
                    f"Departure: {hours:02d}:{mins:02d}"
                )
            ).add_to(m)

    import os, webbrowser
    map_path = os.path.abspath("data/interaktif_harita.html")
    m.save(map_path)
    print(f"\n🗺️  Opening map in browser...")
    webbrowser.open("file://" + map_path)
    display(m)


# ==========================================
# 6. INTERACTIVE SLIDERS
# ==========================================
    # V2 + V3 plots
    from src.plot_results import plot_station_loads, plot_departure_times
    plot_station_loads(x_opt, Q_j, station_names)
    plot_departure_times(x_opt, t_opt, smooth_traffic_curve)

interact_ui = widgets.interact_manual(
    run_optimization,
    ALPHA=widgets.FloatSlider(
        value=0.5, min=0.0, max=5.0, step=0.1,
        description='Traffic Weight (α):',
        style={'description_width': 'initial'}
    ),
    MU=widgets.FloatSlider(
        value=500.0, min=50.0, max=2000.0, step=50.0,
        description='Capacity Penalty (μ):',
        style={'description_width': 'initial'}
    ),
    CROSSING_PENALTY=widgets.FloatSlider(
        value=100.0, min=10.0, max=500.0, step=10.0,
        description='Bosphorus Penalty (×):',
        style={'description_width': 'initial'}
    ),
)


interactive(children=(FloatSlider(value=0.5, description='Traffic Weight (α):', max=5.0, style=SliderStyle(des…